# Calculate 2D electric field with $dr d\Phi$

In [1]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.special import jv, kv
from scipy.special import j1 # Bessel function of the first kind of order 1, for the diffraction pattern calculation
from scipy.optimize import root_scalar
from scipy.constants import c, epsilon_0, mu_0, pi
import scipy.integrate as integrate

In [2]:
from scipy.optimize import brentq

In [3]:
def function_LP01(X, V):
    ""
    # Calculate Y based on the relationship Y^2 = V^2 - X^2
    Y = np.sqrt(V**2 - X**2)
    # Calculate the left-hand side and right-hand side of the equation using the Bessel functions
    lhs = X * jv(1, X) / jv(0, X)
    rhs = Y * kv(1, Y) / kv(0, Y)
    return lhs - rhs

def find_root(V, X_min, X_max):
    # Define a lambda function that takes X as input and uses the previously defined function f with the 
    # fixed V value. This allows us to use this function for root finding.
    function_m1 = lambda X: function_LP01(X, V)

    X_scan = np.linspace(X_min, X_max, 5000)
    F_scan = function_m1(X_scan) # Evaluate the function at the scan points

    bracket = None

    # We take length of scan - 1 because we are checking pairs of points for sign changes, 
    # -1 means we won't go out of bounds when checking F_scan[i + 1]
    for i in range(len(X_scan) - 1): # Loop through the length-1 of X_scan to find a sign change
        if F_scan[i] * F_scan[i + 1] < 0: # Check for a sign change
            x1 = X_scan[i]
            x2 = X_scan[i + 1]
            f1 = F_scan[i]
            f2 = F_scan[i + 1]

            # Skip non-finite values
            if not np.isfinite(f1) or not np.isfinite(f2):
                continue

            # sign chage detected between x1 and x2, we can use this as a bracket for root finding
            if f1*f2 < 0:
                bracket = (x1, x2) # Store the bracket where the sign change occurs
                break # Exit the loop after finding the first sign change

    if bracket is None:
        raise ValueError(f"No valid bracket found for V = {V:.6f}")

    X_root = brentq(function_m1, bracket[0], bracket[1]) # Use the found bracket for root finding
    Y_root = np.sqrt(V**2 - X_root**2)
    return X_root, Y_root


## 2D E-field of misaligned fibre

Start considering the misalignment of the E-field because we can always have a deviation in alignment between the fibre and the pupil. 

In [ ]:
def electric_field_misaligned(r, a, phi):
    # Create a 2D grid in physical coordinates
    x = np.linspace(-3, 3, 1000) * a
    y = np.linspace(-3, 3, 1000) * a
    X_grid, Y_grid = np.meshgrid(x, y)

    rho_2d = np.sqrt(X_grid**2 + Y_grid**2) # Calculate the radial distance from the center of the fiber
    phi_2d = np.arctan2(Y_grid, X_grid) # Calculate the angular coordinate in the 2D grid

    